Chapter 2: Wokring with text

1.1 Tokenizing Text

In [3]:
import os
import urllib.request #downloading things from internet
if not os.path.exists("The-Verdict.txt"):
    url = "https://raw.githubusercontent.com/Srujith-Astro/LLM-From-scratch/refs/heads/main/the%20verdict.txt"
    file_path = "The-Verdict.txt"
    urllib.request.urlretrieve(url,file_path)

In [4]:
with open("The-Verdict.txt","r", encoding="utf-8") as f:
    raw_text = f.read()

In [5]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [6]:
len(raw_text)

20480

In [7]:
import re 
text = "Heloo, how are you? I am fi ne, Tha nks for asking!"
result = re.split(r'(\s)',text)  #this is some library and it is splitting when it sees a space 
print(result)

['Heloo,', ' ', 'how', ' ', 'are', ' ', 'you?', ' ', 'I', ' ', 'am', ' ', 'fi', ' ', 'ne,', ' ', 'Tha', ' ', 'nks', ' ', 'for', ' ', 'asking!']


In [8]:
result = re.split(r'([,.]|\s)',text)
print(result)

['Heloo', ',', '', ' ', 'how', ' ', 'are', ' ', 'you?', ' ', 'I', ' ', 'am', ' ', 'fi', ' ', 'ne', ',', '', ' ', 'Tha', ' ', 'nks', ' ', 'for', ' ', 'asking!']


In [9]:


result = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result


In [10]:
len(preprocessed)

4690

1.2 Converting Tokenized text into token IDs

In [11]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [13]:
vocab = {token:integer for integer, token in enumerate(all_words)}
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

In [14]:
class SimpleTkenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #replace space bfore the specified puntuations
        text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text
    
    # basically string to int(input text), and then that int is mapped(kind of to the exisitng vocab ) then it is reverse convereted into text we return that text

In [15]:
tokenizer = SimpleTkenizerV1(vocab)

In [16]:
text = """"It's the last he time he painted, you know,"
        Mrs. Gisburn   with pardonable pride."""

In [17]:
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 1011, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 1108, 754, 793, 7]


In [18]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he time he painted, you know," Mrs. Gisburn with pardonable pride.'

2.4 Adding special context tokens

In [19]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","|unk|"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
len(vocab.items())

1132

In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('|unk|', 1131)


In [21]:
class SimpleTkenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item if item in self.str_to_int
            else "|unk|" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #replace space bfore the specified puntuations
        text = re.sub(r'\s+([,.?!"()\'])',r'\1',text)
        return text
    
    # basically string to int(input text), and then that int is mapped(kind of to the exisitng vocab ) then it is reverse convereted into text we return that text

In [22]:
tokenizer =SimpleTkenizerV2(vocab)
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he time he painted, you know," Mrs. Gisburn with pardonable pride.'

2.5 Byterpair encoding - we will address unknown words that are not seen in the vocabulary(training)

In [23]:
import tiktoken #-0.12.0 version
tiktoken.__version__

'0.12.0'

In [24]:
tokenizer = tiktoken.get_encoding("gpt2")

In [25]:
tokenizer.encode("Hello workds")

[15496, 670, 9310]

In [26]:
tokenizer.decode(tokenizer.encode("Hello workds"))

'Hello workds'

In [27]:
text = (
    "tony staark was able to build this in a cave!! <asrhh sgafghj with a box of scrapsss"
)
tokenizer.encode(text,allowed_special={"<|endoftext|>"})

[1122,
 88,
 336,
 64,
 668,
 373,
 1498,
 284,
 1382,
 428,
 287,
 257,
 11527,
 3228,
 1279,
 292,
 81,
 12337,
 264,
 70,
 1878,
 456,
 73,
 351,
 257,
 3091,
 286,
 15881,
 824,
 82]

2.6 data sampling with sliding window
 we cant get the model predict everything at once, we need to let it predict some set of things at a time. its like if you have thoughts
you cant really know what you'll be at the end of the sentence(even while speaking) it gets generated as you go,
that is what here is also happening with LLMs you let is predict(literally think) a set number of words/tokens at a time,
ofcourse that is some kind of sampling, which is very interesting, coz we already know signals. generally its 1000s of words/tokens at a time
 but for in our case we are predicting just 4 words at a time dude our training data set a random story from online

In [28]:
with open("The-Verdict.txt","r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5146


In [29]:
enc_sample = enc_text[50:]

In [30]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x:{x}")
print(f"y:      {y}")
# you are targetting your next step, your future

x:[290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [31]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [32]:
import torch

c:\Srujith\Fun\LLM\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [33]:
torch.__version__ #we use this to create embedding from text encoding into tokens. pytorch has Deep learning frameworks to make this embeddings fast

'2.12.0+cpu'

In [34]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize entire texxt
        token_ids = tokenizer.encode(txt,allowed_special ={"<|endoftext|"})

        # use a sliding window to chukc the book into overlapping sequences of max_length
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

            
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx],self.target_ids[idx]

In [41]:
def create_dataloader_v1(txt,batch_size = 4, max_length = 256,
                         stride = 128, shuffle = True, drop_last = True,
                         num_workers=0):
    
    #initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #create dataset
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
    #create data loader
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,
        num_workers = num_workers
    )
    return dataloader 

In [56]:
with open("The-Verdict.txt","r",encoding="utf-8")as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle= False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [57]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [65]:
dataloader = create_dataloader_v1(raw_text,batch_size=8, max_length=6, stride=7, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("outputs:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464,  1807,  3619],
        [  271, 10899,  2138,   257,  7026, 15632],
        [ 2016,   257,   922,  5891,  1576,   438],
        [  340,   373,   645,  1049,  5975,   284],
        [  284,  3285,   326,    11,   287,   262],
        [  286,   465, 13476,    11,   339,   550],
        [  465, 12036,    11,  6405,   257,  5527],
        [   11,   290,  4920,  2241,   287,   257]])
outputs:
 tensor([[  367,  2885,  1464,  1807,  3619,   402],
        [10899,  2138,   257,  7026, 15632,   438],
        [  257,   922,  5891,  1576,   438,   568],
        [  373,   645,  1049,  5975,   284,   502],
        [ 3285,   326,    11,   287,   262,  6001],
        [  465, 13476,    11,   339,   550,  5710],
        [12036,    11,  6405,   257,  5527, 27075],
        [  290,  4920,  2241,   287,   257,  4489]])


2.7 creating Token Embeddibngs
taking the token ID(an integer value) and making that into a vector

In [89]:
inputs_ids = torch.tensor([4,3,5])

In [102]:
# creating embedding layer -> it is part of LLM itself
vocab_size = 6
output_dim = 4

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, output_dim)

In [103]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880],
        [ 0.3486,  0.6603, -0.2196, -0.3792],
        [ 0.7671, -1.1925,  0.6984, -1.4097],
        ...,
        [-0.0265, -0.9674,  0.0700, -0.3521],
        [-1.6701,  0.7788,  0.4831, -2.4027],
        [-1.2396, -2.4954,  0.9409, -2.0232]], requires_grad=True)


In [92]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

In [93]:
embedding_layer(torch.tensor(5))

tensor([-2.8400, -0.7849, -1.4096], grad_fn=<EmbeddingBackward0>)

In [94]:
inputs_ids

tensor([4, 3, 5])

In [95]:
embedding_layer(inputs_ids)

tensor([[-1.1589,  0.3255, -0.6315],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096]], grad_fn=<EmbeddingBackward0>)

In [106]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [107]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,batch_size=8, max_length=  max_length, 
    stride= max_length, shuffle= False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [108]:
print("Token Id:\n", inputs)
print("\n Inputs shape: \n", inputs.shape)

Token Id:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

 Inputs shape: 
 torch.Size([8, 4])


In [109]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [120]:
token_embeddings[2,3]

tensor([-2.4223e-01, -9.7283e-01,  4.4330e-01,  1.0054e-01,  1.2174e+00,
         9.4832e-01,  1.1414e+00,  1.5531e+00,  1.0523e+00, -2.0304e-01,
         2.4346e-01, -2.0583e-01, -6.6821e-01, -7.8717e-01, -5.8568e-01,
         4.9281e-01, -4.3677e-01,  3.1768e-01, -1.7842e+00, -5.4045e-01,
         3.0009e-02, -1.9735e+00, -2.1553e-01,  2.8474e-01, -6.7666e-01,
        -5.3464e-01, -2.0049e-01,  1.1552e+00, -1.3605e+00,  1.3586e+00,
         2.8009e-01, -1.8772e-01,  7.1318e-01,  1.3060e+00, -7.6746e-02,
         2.5952e-01,  1.8869e+00, -7.6815e-01, -8.0938e-01,  1.3445e+00,
         2.5458e-01, -2.7654e-01,  4.6183e-01,  6.0763e-01, -1.0346e+00,
        -6.7541e-01,  7.6640e-01, -5.6152e-01, -7.2827e-01, -4.7371e-01,
        -7.7698e-01,  4.0411e-02, -7.5466e-02,  8.0834e-01, -1.3351e+00,
        -4.4276e-01, -9.4890e-01, -1.2317e-01,  6.7859e-03,  1.7346e+00,
         1.0937e+00,  1.6757e-01,  1.4289e+00,  4.4121e-01,  1.4055e+00,
        -4.1111e-03,  1.2533e+00,  9.8562e-01, -2.1

In [121]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)

In [122]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

In [129]:
pos_embedding_layer.weight

Parameter containing:
tensor([[ 0.9130, -0.6354, -1.2378,  ...,  2.0031,  0.5315, -1.8581],
        [-1.2869,  0.0673, -0.0458,  ..., -1.2039, -1.5753,  2.0907],
        [-0.8893, -0.5778, -0.3387,  ..., -1.1603, -0.0977, -0.1079],
        [-1.0346, -0.8656,  0.6808,  ..., -0.2402,  1.4819,  0.1040]],
       requires_grad=True)

In [131]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [132]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [135]:
input_embeddings = token_embeddings+ pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [136]:
token_embeddings[0] + pos_embeddings

tensor([[ 0.9463, -2.1781,  0.3015,  ...,  2.5684,  0.7118, -3.2170],
        [-1.6020, -1.0708,  0.0397,  ..., -1.6762, -0.6199,  1.4983],
        [-1.1181, -0.5562, -0.5657,  ..., -1.6023,  0.0703,  0.2168],
        [ 0.8006,  0.1483,  0.3251,  ..., -0.2362,  2.8736,  0.4528]],
       grad_fn=<AddBackward0>)

In [137]:
token_embeddings + pos_embeddings

tensor([[[ 0.9463, -2.1781,  0.3015,  ...,  2.5684,  0.7118, -3.2170],
         [-1.6020, -1.0708,  0.0397,  ..., -1.6762, -0.6199,  1.4983],
         [-1.1181, -0.5562, -0.5657,  ..., -1.6023,  0.0703,  0.2168],
         [ 0.8006,  0.1483,  0.3251,  ..., -0.2362,  2.8736,  0.4528]],

        [[ 0.9685,  0.4708, -1.3539,  ...,  0.5897,  0.0108, -1.4075],
         [-0.8306, -0.7951,  0.1380,  ..., -0.5264, -2.6193,  3.3996],
         [-0.1965, -0.9853, -0.3025,  ..., -0.8438,  1.5810, -1.2900],
         [ 0.1682, -1.1416,  0.9581,  ..., -2.1821,  1.6324,  0.2125]],

        [[ 0.2496, -1.2967, -2.0593,  ...,  1.9507,  1.3992, -1.6649],
         [ 0.6663, -0.3908, -1.0128,  ..., -1.4058, -2.0104,  2.1983],
         [-0.2790,  1.5350,  0.5022,  ..., -2.0935, -0.3699, -0.9628],
         [-1.2768, -1.8385,  1.1242,  ..., -1.5650,  0.4222, -0.5530]],

        ...,

        [[ 0.9653, -2.4444,  0.9712,  ...,  3.0978, -0.3677, -1.1480],
         [-3.2686, -0.4761,  2.4350,  ..., -1.8841, -0.16